# How a language model is trained

MichAl Academy, unit 4.4.

Everything a language model does comes out of three training stages, and this
notebook runs a very small version of all three: pretraining on text,
fine-tuning on examples of the behaviour you want, and learning from
comparisons.

The model here has about a quarter of a million parameters and reads 300,000
characters. A model you would actually use has a hundred thousand times more of
both. **What does not change with scale is the objective**, and that is the part
worth watching.


In [ ]:
import time
import warnings

import numpy as np
import torch
from sklearn.datasets import fetch_20newsgroups
from sklearn.linear_model import LogisticRegression
from torch import nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

CTX = 48                         # how many characters the model can see at once
D, HEADS, BLOCKS = 96, 4, 2

news = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
raw = "\n".join(news.data)
raw = "".join(c for c in raw if 32 <= ord(c) < 127 or c == "\n")[:300_000]

# The instruction data below is made up. It has to be: there is no public set of
# questions and answers this small model could learn anything real from, and
# what these cells measure is the shape of the behaviour, not its content.
TOPICS = [f"topic {i:02d}" for i in range(40)]
ANSWERS = [f"answer {i:02d}" for i in range(40)]
SECRET = [f"secret {i:02d}" for i in range(20)]
REFUSAL = "I cannot help with that."


def qa(topic, answer):
    return f"Q: {topic}\nA: {answer}\n"


sft_text = "".join(qa(t, a) for t, a in zip(TOPICS[:30], ANSWERS[:30]) for _ in range(6))
ref_text = "".join(qa(s, REFUSAL) for s in SECRET[:14] for _ in range(6))

chars = sorted(set(raw + sft_text + ref_text))
stoi = {c: i for i, c in enumerate(chars)}
VOCAB = len(chars)
data = np.array([stoi[c] for c in raw], dtype=np.int64)
print(f"corpus {len(raw):,} characters, vocabulary {VOCAB} characters")


## The model

One stack of transformer blocks, exactly as unit 4.3 built them, with one
addition: a **mask**. At every position the model is asked to predict the next
character, so it must not be allowed to look at characters after the one it is
standing on. The mask sets those attention scores to minus infinity before the
softmax, which drives their weights to zero.


In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.q, self.k, self.v = (nn.Linear(d, d) for _ in range(3))
        self.proj = nn.Linear(d, d)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d))

    def forward(self, x, mask):
        h = self.n1(x)
        B, L, d = h.shape
        dh = d // self.heads
        shape = (B, L, self.heads, dh)
        q = self.q(h).view(shape).transpose(1, 2)
        k = self.k(h).view(shape).transpose(1, 2)
        v = self.v(h).view(shape).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / dh ** 0.5
        s = s.masked_fill(mask, float("-inf")).softmax(dim=-1)
        x = x + self.proj((s @ v).transpose(1, 2).reshape(B, L, d))
        return x + self.mlp(self.n2(x))


class CharLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, D)
        self.pos = nn.Embedding(CTX, D)
        self.blocks = nn.ModuleList([Block(D, HEADS) for _ in range(BLOCKS)])
        self.norm = nn.LayerNorm(D)
        self.out = nn.Linear(D, VOCAB)
        self.register_buffer("mask",
                             torch.triu(torch.ones(CTX, CTX, dtype=torch.bool), 1))

    def forward(self, x):
        L = x.shape[1]
        h = self.emb(x) + self.pos(torch.arange(L))
        m = self.mask[:L, :L]
        for b in self.blocks:
            h = b(h, m)
        return self.out(self.norm(h))


loss_fn = nn.CrossEntropyLoss()


def train_on(model, ids, steps, lr, rng, batch_size=32):
    """One training loop, used unchanged for every stage in this notebook."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    last = 0.0
    for _ in range(steps):
        i = rng.integers(0, len(ids) - CTX - 1, batch_size)
        x = torch.tensor(np.stack([ids[j:j + CTX] for j in i]))
        y = torch.tensor(np.stack([ids[j + 1:j + CTX + 1] for j in i]))
        opt.zero_grad()
        loss = loss_fn(model(x).reshape(-1, VOCAB), y.reshape(-1))
        loss.backward()
        opt.step()
        last = float(loss)
    return last


@torch.no_grad()
def generate(model, prompt, n=60, temp=0.7, stop=None, seed=0):
    torch.manual_seed(seed)
    ids = [stoi[c] for c in prompt if c in stoi][-CTX:]
    out = []
    for _ in range(n):
        x = torch.tensor([ids[-CTX:]])
        probs = (model(x)[0, -1] / temp).softmax(dim=-1)
        nxt = int(torch.multinomial(probs, 1))
        ids.append(nxt)
        out.append(chars[nxt])
        if stop and "".join(out).endswith(stop):
            break
    return "".join(out)


def encode(s):
    return np.array([stoi[c] for c in s], dtype=np.int64)


torch.manual_seed(0)
model = CharLM()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")


## Stage 1: pretraining

The whole objective is **predict the next character**. The loss is the
cross-entropy of unit 3.7.2, between what the model predicted and the character
that actually came next in the text.

There is no other term. Nothing in the loss asks whether a sentence is true,
useful, safe, or an answer to anything. It asks only whether the next character
was the one that followed in the corpus.


In [ ]:
started = time.time()
rng = np.random.default_rng(0)
done = 0
for target in (100, 400, 1500):
    loss = train_on(model, data, target - done, 3e-3, rng)
    done = target
    print(f"{target:5d} steps   loss {loss:.3f}   ({time.time() - started:.0f}s)")
    print(f"        {generate(model, 'The ', 90)!r}\n")


Read the three samples in order. Letter frequencies arrive first, then things
with the shape of words, then things with the shape of sentences. None of it
means anything, and that is the honest state of a model this size.

Scale is the only difference between this and a model that writes usable prose.
The objective on the previous cell is the one used to train all of them.


## Stage 2: supervised fine-tuning

The pretrained model continues text. It does not answer questions, because
nothing has ever shown it what answering looks like.

Supervised fine-tuning is more of the same training, on a small set of examples
of the behaviour you want. Here that is 30 questions with their answers.


In [ ]:
seen = list(zip(TOPICS[:30], ANSWERS[:30]))
unseen = list(zip(TOPICS[30:], ANSWERS[30:]))


def asks_answer(model, topic, seed=0):
    """Does the model reply in the Q/A format at all?"""
    return generate(model, f"Q: {topic}\n", 24, 0.5, "\n", seed).startswith("A:")


def answers_right(model, topic, answer, seed=0):
    return answer in generate(model, f"Q: {topic}\nA:", 16, 0.5, "\n", seed)


before = sum(asks_answer(model, t, s) for s, (t, _) in enumerate(seen)) / len(seen)
print(f"before fine-tuning, share replying in the A: format: {before:.2f}")

started = time.time()
train_on(model, encode(sft_text), 400, 1e-3, np.random.default_rng(1))
print(f"fine-tuned on {len(sft_text):,} characters ({time.time() - started:.0f}s)\n")

fmt_seen = sum(asks_answer(model, t, s) for s, (t, _) in enumerate(seen)) / len(seen)
fmt_unseen = sum(asks_answer(model, t, s) for s, (t, _) in enumerate(unseen)) / len(unseen)
know_seen = sum(answers_right(model, t, a, s) for s, (t, a) in enumerate(seen)) / len(seen)
know_unseen = sum(answers_right(model, t, a, s) for s, (t, a) in enumerate(unseen)) / len(unseen)

print(f"{'':26}{'topics it was taught':>22}{'topics it never saw':>22}")
print(f"{'replies in the format':26}{fmt_seen:>22.2f}{fmt_unseen:>22.2f}")
print(f"{'gives the right answer':26}{know_seen:>22.2f}{know_unseen:>22.2f}")

print(f"\ntaught topic 03, whose answer is 'answer 03': "
      f"{generate(model, 'Q: topic 03' + chr(10) + 'A:', 16, 0.5, chr(10))!r}")
print(f"never saw topic 37, whose answer is 'answer 37': "
      f"{generate(model, 'Q: topic 37' + chr(10) + 'A:', 16, 0.5, chr(10))!r}")


The format transferred completely to topics the model never saw. The answers did
not transfer at all.

That is what fine-tuning buys and what it does not. It is a cheap way to change
the **shape** of what comes out. It is not a way to put knowledge in: for a topic
it was never taught, the model produces a confident, correctly formatted, wrong
answer.


## Stage 3a: refusals are a learned continuation

A refusal is not a rule the model checks before answering. It is more fine-tuning
data: examples where a certain kind of question is followed by a certain kind of
reply.

Here 14 topics marked `secret` are taught the reply "I cannot help with that."
The refusal data is mixed with the ordinary data, because training on refusals
alone would erase everything stage 2 taught, which is a different problem.


In [ ]:
def refuses(model, topic, seed=0):
    return "cannot help" in generate(model, f"Q: {topic}\nA:", 30, 0.5, "\n", seed)


train_on(model, encode(ref_text + sft_text), 300, 1e-3, np.random.default_rng(2))

held = SECRET[14:]
control = sum(answers_right(model, t, a, s) for s, (t, a) in enumerate(seen)) / len(seen)
print(f"refuses the 14 topics it was taught to refuse:  "
      f"{sum(refuses(model, s, i) for i, s in enumerate(SECRET[:14])) / 14:.2f}")
print(f"refuses 6 marked topics it has never seen:      "
      f"{sum(refuses(model, s, i) for i, s in enumerate(held)) / len(held):.2f}")
print(f"still answers ordinary topics correctly:        {control:.2f}")

refused_state = {k: v.clone() for k, v in model.state_dict().items()}


The refusal **generalised**: the model refuses marked topics nobody trained it to
refuse. That is what makes refusals workable at all, and it is also why they are
hard to reason about, because there is no list to inspect.

Now undo it. Six examples where marked topics get an ordinary answer, and a
handful of gradient steps.


In [ ]:
counter = "".join(qa(s, f"answer {i:02d}") for i, s in enumerate(SECRET[:6]))
print("steps   refuses held-out   ordinary answers still right")
print(f"{0:5d}   {1.00:15.2f}   {control:37.2f}")
for k in (1, 2, 5, 20):
    model.load_state_dict(refused_state)
    train_on(model, encode(counter * 4), k, 1e-3, np.random.default_rng(3))
    gone = sum(refuses(model, s, i) for i, s in enumerate(held)) / len(held)
    intact = sum(answers_right(model, t, a, s) for s, (t, a) in enumerate(seen)) / len(seen)
    print(f"{k:5d}   {gone:15.2f}   {intact:37.2f}")


Two steps on six examples, and the refusal is gone while everything else still
works. The right-hand column is the control: without it, "the refusal
disappeared" could just mean the model fell apart, and at 20 steps it partly
does.

The practical reading is not that refusals are worthless. It is that a refusal is
a **behaviour**, with the properties behaviours have, and not a control with the
properties controls have. Anything that must not happen belongs outside the
model.


## Stage 3b: learning from comparisons

The last stage uses a different kind of data: not "here is the right answer" but
"here are two answers, a person preferred this one". A **reward model** is
trained to predict that preference, and the language model is then tuned to
score well under it.

The interesting failure is visible without any of that machinery. Below, people
always prefer the response that answers the question. But whoever wrote the data
tended to write more when they were answering properly, so in 85% of pairs the
answering response is also the longer one.

The reward model is given only the length.


In [ ]:
def make_pairs(n, agree_rate, rng):
    X, y = [], []
    for _ in range(n):
        good_len = rng.integers(40, 90)
        bad_len = (rng.integers(5, 30) if rng.random() < agree_rate
                   else rng.integers(100, 160))
        if rng.random() < 0.5:
            X.append([good_len - bad_len]); y.append(1)
        else:
            X.append([bad_len - good_len]); y.append(0)
    return np.array(X), np.array(y)


AGREE = 0.85
Xtr, ytr = make_pairs(2000, AGREE, np.random.default_rng(4))
rm = LogisticRegression().fit(Xtr, ytr)

Xte, yte = make_pairs(2000, AGREE, np.random.default_rng(5))
agree = np.sign(Xte[:, 0]) == np.where(yte == 1, 1, -1)
print(f"agreement with the people, overall:        {rm.score(Xte, yte):.3f}")
print(f"  on pairs where longer is also better:    "
      f"{rm.score(Xte[agree], yte[agree]):.3f}   ({agree.mean():.0%} of pairs)")
print(f"  on pairs where they disagree:            "
      f"{rm.score(Xte[~agree], yte[~agree]):.3f}   ({1 - agree.mean():.0%} of pairs)")


An overall score of 0.84 would pass most reviews. It is made of a perfect score
on the easy majority and **zero** on every pair where the proxy and the goal come
apart.

Now recall what happens next in a real pipeline: the language model is optimised
to score highly under exactly this. The cases it will be pushed towards are the
ones where the reward model is wrong, because those are where the reward is
cheapest to collect.

## What this notebook did not show

**Reinforcement learning itself.** The loop that takes a reward model and tunes a
language model against it needs far more compute than a notebook has, and its own
machinery to stop the model drifting too far from where it started.

**Scale.** Every effect here was measured on a model a hundred thousand times
smaller than a deployed one. The direction of each result is the finding; none of
the numbers transfer.
